In [2]:
import os
import pandas as pd
import plotly.graph_objects as go
from typing import List, Optional, Dict
import numpy as np

def find_config_file(folder_path: str) -> Optional[str]:
    """
    Finds a configuration file (ending with .txt) within the 'configs' subfolder.
    """
    config_dir = os.path.join(folder_path, 'configs')
    if not os.path.isdir(config_dir):
        return None
    for item in os.listdir(config_dir):
        if item.endswith('.txt'):
            return os.path.join(config_dir, item)
    return None

def parse_config(file_path: str) -> Dict[str, str]:
    """
    Parses a 'key = value', 'key value', or 'key: value' configuration file into a dictionary.
    """
    params = {}
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                
                separator = None
                if ':' in line:
                    separator = ':'
                elif '=' in line:
                    separator = '='

                if separator:
                    parts = line.split(separator, 1)
                else:
                    parts = line.split(None, 1)

                if len(parts) == 2:
                    key, value = parts
                    params[key.strip().lower()] = value.strip()
    except FileNotFoundError:
        print(f"Config file not found: {file_path}")
    except Exception as e:
        print(f"Error parsing config file {file_path}: {e}")
    return params

In [3]:
import re
import pandas as pd

# --- Configuration for G2 vs NS3 Comparison ---

# Define the base folder containing all the run directories from all workloads
# We will process all workloads ('toy_all_to_all_one_collective', 'toy_all_reduce_one_collective', 'model')
base_comparison_folder = '/app/astra-sim/upc/output/comparison_run/FoldedClos/T5_Base_grouped_comp'

# Choose what to plot: 'avg' for mean, or 'max' for maximum value
comparison_plot_metric = 'max'  # Can be 'avg' or 'max'

# --- Data Collection Logic ---

all_run_folders = []
for workload_folder in os.listdir(base_comparison_folder):
    workload_path = os.path.join(base_comparison_folder, workload_folder)
    if os.path.isdir(workload_path):
        run_folders = [os.path.join(workload_path, d) for d in os.listdir(workload_path) if os.path.isdir(os.path.join(workload_path, d))]
        all_run_folders.extend(run_folders)

comparison_results = []

# Regex to extract topology index
topo_idx_regex = re.compile(r'topology(\d+)(?:\.json)?$')

cc_modes = {0: "PFC", 1: "DCQCN", 3: "HPCC", 7: "TIMELY", 8: "DCTCP", 10: "HPCC-PINT"}

def parse_runtime(time_str: str) -> float:
    """Parses 'H:MM:SS.ffffff' into seconds."""
    if not time_str:
        return 0.0
    try:
        parts = time_str.split(':')
        h = int(parts[0])
        m = int(parts[1])
        s = float(parts[2])
        return h * 3600 + m * 60 + s
    except (ValueError, IndexError):
        return 0.0

for folder in sorted(all_run_folders):
    run_summary_path = os.path.join(folder, 'run_summary.txt')
    if not os.path.exists(run_summary_path):
        continue

    # 1. Parse run_summary.txt to identify sim_type and topology
    summary_params = parse_config(run_summary_path)
    
    workload_name = summary_params.get('collective', 'N/A').strip()
    npu_count = summary_params.get('npus count', 'N/A')
    total_runtime_str = summary_params.get('total runtime', '0:0:0.0')
    execution_time_sec = parse_runtime(total_runtime_str)

    # Process 'analytical_unaware' separately as it's topology-independent
    if os.path.exists(os.path.join(folder, 'analytical_unaware')):
        timing_file = None
        sim_output_dir = os.path.join(folder, 'analytical_unaware')
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break
        
        if timing_file:
            try:
                df = pd.read_csv(timing_file)
                time_col = 'callback_tick'
                if time_col in df.columns:
                    elapsed_times = df[time_col].dropna()
                    if not elapsed_times.empty:
                        comparison_results.append({
                            'workload': workload_name,
                            'npu_count': npu_count,
                            'topo_index': -1,  # Use -1 to indicate topology independence
                            'sim_type': 'Analytical Unaware',
                            'run_name': 'Analytical Unaware',
                            'avg_time': elapsed_times.mean(),
                            'max_time': elapsed_times.max(),
                            'std_dev': elapsed_times.std(),
                            'execution_time': execution_time_sec,
                            'path': os.path.basename(folder),
                            'topology_file': 'N/A'
                        })
            except Exception as e:
                print(f"Error processing analytical_unaware in {folder}: {e}")

    # Process topology-dependent simulations (G2, NS3)
    sim_type = None
    topo_file = None
    if os.path.exists(os.path.join(folder, 'g2')):
        sim_type = 'G2'
        topo_file = summary_params.get('g2 topology file override', 'N/A')
    elif os.path.exists(os.path.join(folder, 'ns3')):
        sim_type = 'NS3'
        topo_file = summary_params.get('ns3 topology file override', 'N/A')

    if not sim_type or not topo_file or 'all_paths' in topo_file:
        continue

    # Extract topology index
    match = topo_idx_regex.search(topo_file)
    if not match:
        continue
    topo_index = int(match.group(1))

    # 2. Get timing data
    timing_file = None
    sim_output_dir = None
    if sim_type == 'G2':
        sim_output_dir = os.path.join(folder, 'g2')
    elif sim_type == 'NS3':
        sim_output_dir = os.path.join(folder, 'ns3')

    if sim_output_dir and os.path.isdir(sim_output_dir):
        for f in os.listdir(sim_output_dir):
            if 'trace_matched_timing.csv' in f:
                timing_file = os.path.join(sim_output_dir, f)
                break
    
    if not timing_file:
        continue

    try:
        df = pd.read_csv(timing_file)
        if sim_type == 'NS3':
            df = df[df['node_name'] != 'dummy_node'].copy()
        
        time_col = 'callback_tick'
        if time_col not in df.columns:
            continue
            
        elapsed_times = df[time_col].dropna()
        if elapsed_times.empty:
            continue

        # 3. Create a descriptive name and store results
        run_name = f"{sim_type}"
        if sim_type == 'NS3':
            ns3_config_file = find_config_file(folder)
            if ns3_config_file:
                ns3_params = parse_config(ns3_config_file)
                run_name = (
                    f"NS3 (cc:{cc_modes.get(int(ns3_params.get('cc_mode', 'N/A')), 'N/A')}, "
                    f"win:{ns3_params.get('has_win', 'N/A')}, "
                    f"adapt:{ns3_params.get('var_win', 'N/A')}, "
                    f"buf:{ns3_params.get('buffer_size', 'N/A')}, "
                    f"size:{ns3_params.get('packet_payload_size', 'N/A')})"
                )

        comparison_results.append({
            'workload': workload_name,
            'npu_count': npu_count,
            'topo_index': topo_index,
            'sim_type': sim_type,
            'run_name': run_name,
            'avg_time': elapsed_times.mean(),
            'max_time': elapsed_times.max(),
            'std_dev': elapsed_times.std(),
            'execution_time': execution_time_sec,
            'path': os.path.basename(folder),
            'topology_file': topo_file
        })

    except Exception as e:
        print(f"Error processing {folder}: {e}")

# --- Plotting Logic ---

if comparison_results:
    comp_df = pd.DataFrame(comparison_results)
    
    # Determine which column to use for plotting
    if comparison_plot_metric == 'max':
        y_col = 'max_time'
        y_axis_title = "Maximum Time (ns)"
    else: # Default to 'avg'
        y_col = 'avg_time'
        y_axis_title = "Average Time (ns)"

    workloads = comp_df['workload'].unique()
    topology_name = os.path.basename(base_comparison_folder)
    
    for wl in sorted(workloads):
        workload_df = comp_df[comp_df['workload'] == wl]
        
        # For this workload, find the single analytical unaware time, if it exists.
        au_runs = workload_df[workload_df['sim_type'] == 'Analytical Unaware']
        au_time = None
        if not au_runs.empty:
            au_time = au_runs.iloc[0][y_col]

        for topo_idx in sorted(workload_df['topo_index'].unique()):
            # Skip the placeholder index used for analytical_unaware
            if topo_idx == -1:
                continue

            group_df = workload_df[workload_df['topo_index'] == topo_idx].copy()
            
            if group_df.empty:
                continue

            # Separate G2 and NS3 for plotting
            g2_runs = group_df[group_df['sim_type'] == 'G2']
            ns3_runs = group_df[group_df['sim_type'] == 'NS3'].sort_values(by=y_col)
            
            if ns3_runs.empty:
                continue # Don't plot if there's no NS3 data to compare against

            plot_title = f'G2 vs NS3 Comparison for Workload: "{wl}", Static routing: {topo_idx}'

            fig = go.Figure()

            # Add NS3 runs as bars
            fig.add_trace(go.Bar(
                x=ns3_runs['run_name'],
                y=ns3_runs[y_col],
                name='NS3 Runs',
                marker_color='rgb(55, 83, 109)',
                text=ns3_runs[y_col].apply(lambda x: f'{x/1e9:.4f} s'),
                textposition='outside'
            ))

            # --- Speedup Calculation & Annotation ---
            fastest_ns3_time = ns3_runs[y_col].min() # Use fastest NS3 for speedup baseline
            npu_count_val = group_df['npu_count'].iloc[0] if not group_df.empty else 'N/A'
            
            g2_time = None
            g2_sim_time_error_text = "G2: N/A (No G2 run)"
            if not g2_runs.empty:
                g2_time = g2_runs.iloc[0][y_col]
                # Calculate percentage error relative to fastest NS3
                sim_time_error_pct = ((g2_time - fastest_ns3_time) / fastest_ns3_time) * 100
                g2_sim_time_error_text = f"G2: {sim_time_error_pct:+.2f}%"
                fig.add_hline(
                    y=g2_time, 
                    line_dash="dot",
                    annotation_text=f"G2 Time: {g2_time/1e9:.4f} s", 
                    annotation_position="top right",
                    line_color="red",
                    annotation=dict(font=dict(color="white", size=12), bgcolor="red", borderpad=4)
                )

            au_sim_time_error_text = "Unaware: N/A"
            if au_time is not None:
                # Calculate percentage error relative to fastest NS3
                sim_time_error_pct = ((au_time - fastest_ns3_time) / fastest_ns3_time) * 100
                au_sim_time_error_text = f"Unaware: {sim_time_error_pct:+.2f}%"
                fig.add_hline(
                    y=au_time, 
                    line_dash="dash",
                    annotation_text=f"Analytical Unaware: {au_time/1e9:.4f} s", 
                    annotation_position="bottom right",
                    line_color="green",
                    annotation=dict(font=dict(color="white", size=12), bgcolor="green", borderpad=4)
                )

            # --- Execution Time Speedup Calculation ---
            fastest_ns3_exec_time = ns3_runs['execution_time'].min()

            g2_exec_speedup_text = "N/A"
            if not g2_runs.empty:
                g2_exec_time = g2_runs.iloc[0]['execution_time']
                if g2_exec_time > 0:
                    exec_speedup = fastest_ns3_exec_time / g2_exec_time
                    g2_exec_speedup_text = f"{exec_speedup:.2f}x"

            au_exec_speedup_text = "N/A"
            if not au_runs.empty:
                au_exec_time = au_runs.iloc[0]['execution_time']
                if au_exec_time > 0:
                    exec_speedup = fastest_ns3_exec_time / au_exec_time
                    au_exec_speedup_text = f"{exec_speedup:.2f}x"

            # Construct the summary text
            summary_text = (
                f"<b>Summary</b><br>"
                f"--------------------<br>"
                f"<b>Topology:</b> {topology_name}<br>"
                f"<b>NPU Nodes:</b> {npu_count_val}<br>"
                f"<b>Static Routing:</b> {topo_idx}<br>"
                f"--------------------<br>"
                f"<b>Execution Time Error vs Fastest NS3 (%):</b><br>"
                f"- {g2_sim_time_error_text}<br>"
                f"- {au_sim_time_error_text}<br>"
                f"--------------------<br>"
                f"<b>Sim Time Speedup vs Fastest NS3:</b><br>"
                f"- G2: {g2_exec_speedup_text}<br>"
                # f"- Unaware: {au_exec_speecdup_text}"
            )

            fig.add_annotation(
                text=summary_text,
                align='left',
                showarrow=False,
                xref='paper',
                yref='paper',
                x=1.29,
                y=0.8,
                bordercolor="black",
                borderwidth=1,
                bgcolor="rgba(255, 255, 255, 0.8)"
            )

            fig.update_layout(
                title=plot_title,
                xaxis_title="Run Configuration",
                yaxis_title=y_axis_title,
                xaxis={'tickangle': -60},
                template='plotly_white',
                height=700,
                width=1400,
                margin=dict(b=350, r=300), # Increased right margin for the text box
                legend=dict(x=1.05, y=1.0), # Adjust legend position
                showlegend=True
            )
            # fig.show()
else:
    print("\nNo comparison results to plot.")

# Display the full data table
if comparison_results:
    print("\n--- Full Comparison Data ---")
    # Reorder columns for better readability
    display_cols = [
        'workload', 'topo_index', 'sim_type', 'run_name', 
        y_col, 'std_dev', 'execution_time', 'npu_count', 'path', 'topology_file'
    ]
    # Ensure all columns exist before trying to display them
    final_cols = [c for c in display_cols if c in comp_df.columns]
    # with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
    #     display(comp_df.sort_values(by=['workload', 'topo_index', y_col])[final_cols])



--- Full Comparison Data ---


In [4]:
import plotly.graph_objects as go
import pandas as pd
from scipy.stats import spearmanr

# This cell creates plots to compare the ordering of max_time across different simulators
# for workloads starting with 'basic_model_'. It also calculates and displays the
# Spearman's rank correlation within the plots.

if 'comp_df' in locals() and not comp_df.empty:
    # 1. Filter for the relevant workloads
    model_df = comp_df[comp_df['workload'].str.contains('T5_Base_multiple_', na=False)].copy()

    # 2. For each workload and topology, find the fastest NS3 run and the corresponding G2/AU times.
    comparison_data = []
    for (wl, topo), group in model_df.groupby(['workload', 'topo_index']):
        if topo == -1:  # Skip analytical-only entries in this grouping
            continue

        ns3_runs = group[group['sim_type'] == 'NS3']
        g2_runs = group[group['sim_type'] == 'G2']
        # Find the corresponding analytical unaware run for the same workload
        au_runs = model_df[(model_df['workload'] == wl) & (model_df['sim_type'] == 'Analytical Unaware')]

        if not ns3_runs.empty:
            ground_truth_ns3_time = ns3_runs['max_time'].min()
            g2_time = g2_runs['max_time'].iloc[0] if not g2_runs.empty else None
            au_time = au_runs['max_time'].iloc[0] if not au_runs.empty else None
            
            comparison_data.append({
                'workload': wl,
                'topo_index': topo,
                'NS3': ground_truth_ns3_time,
                'G2': g2_time,
                'Analytical Unaware': au_time
            })

    if comparison_data:
        plot_df = pd.DataFrame(comparison_data)

        # 3. Create plots for each topology index found
        for topo_idx in sorted(plot_df['topo_index'].unique()):
            topo_specific_df = plot_df[plot_df['topo_index'] == topo_idx].copy()
            
            # 4. Sort workloads based on the ground truth (NS3) max_time
            topo_specific_df = topo_specific_df.sort_values(by='NS3').reset_index(drop=True)
            topo_specific_df['short_workload'] = topo_specific_df['workload'].str.replace('T5_Base_grouped_comp', '', regex=False)

            # 5. Calculate Spearman's Rank Correlation
            cleaned_df = topo_specific_df.dropna(subset=['NS3', 'G2', 'Analytical Unaware'])
            correlation_text = "<b>Spearman's Rank Correlation (ρ)</b><br>--------------------<br>"
            if len(cleaned_df) < 2:
                correlation_text += "Not enough data to calculate."
            else:
                g2_corr, g2_p = spearmanr(cleaned_df['NS3'], cleaned_df['G2'])
                au_corr, au_p = spearmanr(cleaned_df['NS3'], cleaned_df['Analytical Unaware'])
                correlation_text += (
                    f"- G2 vs NS3: {g2_corr:.3f} (p={g2_p:.3f})<br>"
                    f"- Unaware vs NS3: {au_corr:.3f} (p={au_p:.3f})"
                )

            sim_types = ['NS3', 'G2', 'Analytical Unaware']
            colors = {'NS3': 'blue', 'G2': 'red', 'Analytical Unaware': 'green'}

            # --- 7. Generate Bar Plot ---
            fig_bar = go.Figure()
            for sim in sim_types:
                if sim in topo_specific_df.columns:
                    fig_bar.add_trace(go.Bar(
                        x=topo_specific_df['short_workload'], y=topo_specific_df[sim],
                        name=sim, marker_color=colors[sim]
                    ))

            fig_bar.add_annotation(
                text=correlation_text, align='left', showarrow=False,
                xref='paper', yref='paper', x=0.02, y=0.98,
                bordercolor="black", borderwidth=1, bgcolor="rgba(255, 255, 255, 0.8)"
            )
            fig_bar.update_layout(
                barmode='group',
                title=f'Bar Plot: Max Time Comparison (Sorted by NS3) - Topology Index: {topo_idx}',
                xaxis_title="Workload (basic_model_...)", yaxis_title="Maximum Time (ns)",
                template='plotly_white', height=600, width=1200,
                xaxis={'tickangle': -45}, legend_title="Simulator"
            )
            fig_bar.show()

            # --- Add Bump Chart for Rankings ---
            fig_bump = go.Figure()
            
            # Compute ranks (lower time = better rank, i.e., lower rank number)
            topo_specific_df['NS3_rank'] = topo_specific_df['NS3'].rank(method='dense', ascending=True)
            topo_specific_df['G2_rank'] = topo_specific_df['G2'].rank(method='dense', ascending=True)
            topo_specific_df['Analytical Unaware_rank'] = topo_specific_df['Analytical Unaware'].rank(method='dense', ascending=True)
            
            # Add a line for each workload showing rank across simulators
            for idx, row in topo_specific_df.iterrows():
                fig_bump.add_trace(go.Scatter(
                    x=['G2', 'NS3', 'Analytical Unaware'],
                    y=[row['G2_rank'], row['NS3_rank'], row['Analytical Unaware_rank']],
                    mode='lines+markers',
                    name=row['short_workload'],
                    line=dict(color='black', width=2), # remove 'black' to have color
                    marker=dict(size=8)
                ))
            
            fig_bump.update_layout(
                title=f'Bump Chart: Rankings by Max Time - Topology Index: {topo_idx}',
                xaxis_title='Simulator',
                yaxis_title='Rank (1 = Best)',
                yaxis=dict(autorange='reversed'),  # Rank 1 at the top
                template='plotly_white',
                height=600,
                width=1200,
                showlegend=True
            )
            fig_bump.show()

    else:
        print("No 'basic_model' data found to plot.")
else:
    print("Please run the preceding cells to generate the 'comp_df' DataFrame first.")


In [49]:
import plotly.graph_objects as go
import pandas as pd
from scipy.stats import spearmanr

# This cell creates plots to compare the ordering of max_time across different simulators
# for workloads starting with 'basic_model_'. It groups the data by 'topology_file'.

if 'comp_df' in locals() and not comp_df.empty:
    # 1. Filter for the relevant workloads
    model_df = comp_df[comp_df['workload'].str.contains('all_to_all_', na=False)].copy()

    # 2. For each workload and topology file, find the fastest NS3 run and the corresponding G2/AU times.
    comparison_data = []
    # Group by workload and the specific topology file path
    for (wl, topo_file), group in model_df.groupby(['workload', 'topology_file']):
        # Skip analytical-only entries which have 'N/A' as topology_file
        if topo_file == 'N/A':
            continue

        ns3_runs = group[group['sim_type'] == 'NS3']
        g2_runs = group[group['sim_type'] == 'G2']
        # Find the corresponding analytical unaware run for the same workload
        au_runs = model_df[(model_df['workload'] == wl) & (model_df['sim_type'] == 'Analytical Unaware')]

        if not ns3_runs.empty:
            ground_truth_ns3_time = ns3_runs['max_time'].min()
            g2_time = g2_runs['max_time'].iloc[0] if not g2_runs.empty else None
            au_time = au_runs['max_time'].iloc[0] if not au_runs.empty else None
            
            comparison_data.append({
                'workload': wl,
                'topology_file': topo_file,
                'NS3': ground_truth_ns3_time,
                'G2': g2_time,
                'Analytical Unaware': au_time
            })

    if comparison_data:
        plot_df = pd.DataFrame(comparison_data)

        # 3. Create plots for each unique topology file found
        for topo_file in sorted(plot_df['topology_file'].unique()):
            topo_specific_df = plot_df[plot_df['topology_file'] == topo_file].copy()
            
            # 4. Sort workloads based on the ground truth (NS3) max_time
            topo_specific_df = topo_specific_df.sort_values(by='NS3').reset_index(drop=True)
            topo_specific_df['short_workload'] = topo_specific_df['workload'].str.replace('basic_model_', '', regex=False)

            # 5. Calculate Spearman's Rank Correlation
            cleaned_df = topo_specific_df.dropna(subset=['NS3', 'G2', 'Analytical Unaware'])
            correlation_text = "<b>Spearman's Rank Correlation (ρ)</b><br>--------------------<br>"
            if len(cleaned_df) < 2:
                correlation_text += "Not enough data to calculate."
            else:
                g2_corr, g2_p = spearmanr(cleaned_df['NS3'], cleaned_df['G2'])
                au_corr, au_p = spearmanr(cleaned_df['NS3'], cleaned_df['Analytical Unaware'])
                correlation_text += (
                    f"- G2 vs NS3: {g2_corr:.3f} (p={g2_p:.3f})<br>"
                    f"- Unaware vs NS3: {au_corr:.3f} (p={au_p:.3f})"
                )

            sim_types = ['NS3', 'G2', 'Analytical Unaware']
            colors = {'NS3': 'blue', 'G2': 'red', 'Analytical Unaware': 'green'}

            # --- 6. Generate Bar Plot ---
            fig_bar = go.Figure()
            for sim in sim_types:
                if sim in topo_specific_df.columns:
                    fig_bar.add_trace(go.Bar(
                        x=topo_specific_df['short_workload'], y=topo_specific_df[sim],
                        name=sim, marker_color=colors[sim]
                    ))

            fig_bar.add_annotation(
                text=correlation_text, align='left', showarrow=False,
                xref='paper', yref='paper', x=0.02, y=0.98,
                bordercolor="black", borderwidth=1, bgcolor="rgba(255, 255, 255, 0.8)"
            )
            
            plot_title = (
                f'Bar Plot: Max Time Comparison (Sorted by NS3)<br>'
                f'Topology File: {os.path.basename(topo_file)}'
            )
            
            fig_bar.update_layout(
                barmode='group',
                title=plot_title,
                xaxis_title="Workload (basic_model_...)", yaxis_title="Maximum Time (ns)",
                template='plotly_white', height=600, width=1200,
                xaxis={'tickangle': -45}, legend_title="Simulator"
            )
            fig_bar.show()

    else:
        print("No 'basic_model' data found to plot.")
else:
    print("Please run the preceding cells to generate the 'comp_df' DataFrame first.")

In [ ]:
toy_all_to_all_one_collective	4	G2	G2	10119040019	2.039536e+08	0.759469	16	run_20251201_123752_299ms	/app/astra-sim/upc/configuration/g2/topologies...
106	toy_all_to_all_one_collective	4	NS3	NS3 (cc:DCQCN, win:1, adapt:1, buf:1, size:1500)	10394304020	3.481082e+08	240.231722	16	run_20251201_130227_158ms	/app/astra-sim/upc/configuration/ns3/topologie

In [59]:
import pandas as pd
import plotly.graph_objects as go

# --- File Paths ---
g2_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251201_123752_299ms/g2/all_to_all_trace_matched_timing.csv'
ns3_file = '/app/astra-sim/upc/output/comparison_run/FoldedClos/toy_all_to_all_one_collective/run_20251201_130227_158ms/ns3/all_to_all_trace_matched_timing.csv'

# --- Data Loading and Preparation ---
try:
    # Load the data from CSV files
    df_ns3 = pd.read_csv(ns3_file)
    df_g2 = pd.read_csv(g2_file)

    # Select relevant columns and rename for merging
    ns3_times = df_ns3[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'ns3_time'})
    g2_times = df_g2[['sys_id', 'callback_tick']].rename(columns={'callback_tick': 'g2_time'})

    # Merge the two dataframes on 'sys_id' to align the times
    merged_df = pd.merge(ns3_times, g2_times, on='sys_id')

    # --- Plotting ---
    fig = go.Figure()

    # Add a scatter plot of G2 vs NS3 times
    fig.add_trace(go.Scatter(
        x=merged_df['ns3_time'],
        y=merged_df['g2_time'],
        mode='markers',
        name='Time Comparison (per NPU)',
        marker=dict(color='blue'),
        text=merged_df['sys_id'].apply(lambda x: f'NPU {x}'),
        hoverinfo='text+x+y'
    ))

    # Add a y=x line for reference (where times would be identical)
    min_val = min(merged_df['ns3_time'].min(), merged_df['g2_time'].min())
    max_val = max(merged_df['ns3_time'].max(), merged_df['g2_time'].max())
    fig.add_trace(go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        name='Ideal Match (y=x)',
        line=dict(color='red', dash='dash')
    ))

    # Update layout for clarity
    fig.update_layout(
        title='G2 vs NS3 Simulation Time Comparison',
        xaxis_title='NS3 Time (ns)',
        yaxis_title='G2 Time (ns)',
        template='plotly_white',
        width=800,
        height=800,
        showlegend=True
    )

    fig.show()

except FileNotFoundError as e:
    print(f"Error: Could not find a file. {e}")
except Exception as e:
    print(f"An error occurred: {e}")
